# Build Corpora with sentence-corpora

This notebook builds and pickles the stylometric corpora using the sentence-corpora package.

## Stages

1. **Setup** - Import libraries and define paths
2. **Load Data** - Read masked files from data/
3. **Build TwoLevelCorpus** - Create Sentence objects with metadata
4. **Split Corpora** - Divide into train and test sets, holding out X_machabeis
5. **Save Corpora** - Pickle corpora for later use

## Stage 1: Setup

Import required libraries and define paths.

In [3]:
from __future__ import annotations

import pickle
import random
import re
from collections import defaultdict
from pathlib import Path

from sentence_corpora import Sentence, TwoLevelCorpus

## Stage 2: Load Data

Read all masked files and extract metadata from filenames.

In [5]:
DATA_DIR: Final = Path("data_processed_machabeis")


def parse_filename(path: Path) -> tuple[str, str]:
    """Parse '<authorcode>_<workcode>_sentences.quesplit.masked.txt' into (author, work).

    Handles extra whitespace and underscores by splitting on '_sentences.quesplit.masked.txt'.
    """
    name = path.name
    # Remove the _sentences.quesplit.masked.txt suffix
    name = re.sub(r"_sentences\.quesplit\.masked\.txt$", "", name)
    # Split on first underscore to separate authorcode and workcode
    parts = name.split("_", 1)
    if len(parts) != 2:
        raise ValueError(
            f"Expected '<authorcode>_<workcode>' in filename, got: {path.name!r}"
        )
    return parts[0], parts[1]


def build_corpus(data_dir: Path = DATA_DIR) -> TwoLevelCorpus:
    """Read all *_sentences.quesplit.masked.txt files and return a TwoLevelCorpus."""
    sentences = []
    masked_files = sorted(data_dir.glob("*_sentences.quesplit.masked.txt"))

    for path in masked_files:
        author, work = parse_filename(path)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    sentences.append(
                        Sentence(
                            text=line,
                            metadata={
                                "author": author,
                                "work": work,
                            },
                        )
                    )

    return TwoLevelCorpus(sentences, levels=["author", "work"])

In [6]:
corpus = build_corpus()
print(f"Built corpus with {len(corpus)} sentences")
print(f"Authors: {corpus.get_unique_values('author')}")
unique_works = set((s.metadata["author"], s.metadata["work"]) for s in corpus)
print(f"Works: {len(unique_works)} total")

for author, work in corpus.get_unique_tuples():
    print(f"{author}_{work}")

Built corpus with 51144 sentences
Authors: ['AI', 'AeC', 'AeP', 'B', 'BC', 'DC', 'E', 'FB', 'GAp', 'GAr', 'GB', 'GC', 'GM', 'GP', 'GVS', 'GiP', 'H', 'HA', 'HP', 'IDG', 'IH', 'II', 'NLC', 'OM', 'OSE', 'RC', 'RQ', 'RT', 'SG', 'U1', 'U10', 'U11', 'U2', 'U3', 'U4', 'U5', 'U6', 'U7', 'U8', 'U9', 'X']
Works: 43 total
AI_anticlaudianus
AeC_medicaminum
AeC_viaticus
AeP_karolinus
B_judas
BC_contemptu
DC_mathildis
E_biblia
FB_nuptis
GAp_roberti
GAr_philippis
GB_glossarium
GC_alexandreis
GM_merlini
GP_ligurinus
GVS_poetria
GiP_historia
H_maria
HA_planctus
HP_maiolichinus
IDG_gramatice
IH_architrenius
II_ylias
NLC_passio
OM_ernestus
OSE_doctrina
RC_tancredi
RQ_catherinae
RT_mauri
SG_ecclesiasten
U1_canticamariae
U10_karolellus
U11_katherinae
U2_liberesdre
U3_generatione
U4_dialogi
U5_britanniae
U5_brutus
U6_roudlieb
U7_flosmedicinae
U8_friderici
U9_vetula
X_machabeis


## Stage 3: Split Corpora

Split corpus into train and test sets, holding out the work X_machabeis.

Returns (train, test, holdout) where:
  - train contains ~70% of tokens from every work except X_machabeis (configurable via train_pct)
  - test contains the remaining ~30%
  - De Machabeis held out completely (not used in train or test)

In [8]:
def train_test_split(
    corpus: TwoLevelCorpus, seed: int = 42, train_pct: float = 0.5
) -> tuple[TwoLevelCorpus, TwoLevelCorpus, TwoLevelCorpus]:
    """Split corpus into train, test, and a hold-out corpus for X_machabeis.

    Args:
        corpus: The corpus to split.
        seed: Random seed for reproducibility.
        train_pct: Fraction of tokens to allocate to train (0-1). Test gets 1-train_pct.

    Returns (train, test, holdout) where:
      - holdout contains all sentences from X_machabeis
      - train contains ~train_pct of tokens from every other work
      - test contains the remaining ~1-train_pct tokens
    """
    holdout_sentences = []
    test_sentences = []
    train_sentences = []

    rng = random.Random(seed)

    # Group sentences by (author, work)
    by_work = defaultdict(list)
    for s in corpus:
        key = (s.author, s.work)
        by_work[key].append(s)

    for (author, work), sentences in by_work.items():
        if work == "machabeis" and author == "X":
            holdout_sentences.extend(sentences)
            continue
        # Shuffle, then greedily add sentences to train until ~train_pct of tokens reached
        shuffled = list(sentences)
        rng.shuffle(shuffled)

        total_tokens = sum(len(s.text.split()) for s in shuffled)
        target = total_tokens * train_pct

        accumulated = 0
        cutoff = len(shuffled)
        for i, s in enumerate(shuffled):
            if accumulated >= target:
                cutoff = i
                break
            accumulated += len(s.text.split())

        train_sentences.extend(shuffled[:cutoff])
        test_sentences.extend(shuffled[cutoff:])

    return (
        TwoLevelCorpus(train_sentences, levels=["author", "work"]),
        TwoLevelCorpus(test_sentences, levels=["author", "work"]),
        TwoLevelCorpus(holdout_sentences, levels=["author", "work"]),
    )

In [9]:
train_corpus, test_corpus, machabeis_corpus = train_test_split(corpus, train_pct=0.7)
print(f"Train corpus: {len(train_corpus)} sentences")
print(f"Test corpus:  {len(test_corpus)} sentences")
print(f"Holdout corpus (X_machabeis): {len(machabeis_corpus)} sentences")
print()

Train corpus: 35578 sentences
Test corpus:  15353 sentences
Holdout corpus (X_machabeis): 213 sentences



In [10]:
train_corpus.stats()

Author      Works    Sentences    Tokens    Mean Len
--------  -------  -----------  --------  ----------
AI              1        1,076    23,023        21.4
AeC             2        1,183    34,577        29.2
AeP             1          439    13,802        31.4
B               1          290     4,711        16.2
BC              1        1,890    16,325         8.6
DC              1          895    15,048        16.8
E               1          351     7,810        22.3
FB              1        1,882    24,963        13.3
GAp             1          953    14,034        14.7
GAr             1        1,264    26,532        21
GB              1        1,287    11,720         9.1
GC              1        1,434    28,367        19.8
GM              1          441     7,936        18
GP              1        1,197    33,753        28.2
GVS             1          947    11,608        12.3
GiP             1          746    13,405        18
H               1          300     4,625        15.4

In [11]:
test_corpus.stats()

Author      Works    Sentences    Tokens    Mean Len
--------  -------  -----------  --------  ----------
AI              1          470     9,861        21
AeC             2          509    14,745        29
AeP             1          190     5,803        30.5
B               1          111     1,991        17.9
BC              1          830     6,994         8.4
DC              1          398     6,447        16.2
E               1          160     3,325        20.8
FB              1          775    10,682        13.8
GAp             1          422     5,978        14.2
GAr             1          532    11,362        21.4
GB              1          564     5,016         8.9
GC              1          635    12,150        19.1
GM              1          195     3,398        17.4
GP              1          527    14,412        27.3
GVS             1          395     4,954        12.5
GiP             1          313     5,738        18.3
H               1          110     1,963        17

In [12]:
train_1000 = train_corpus.chunk(1000)
train_1000.stats()

Author      Works    Sentences    Tokens    Mean Len
--------  -------  -----------  --------  ----------
AI             22        1,076    23,023        21.4
AeC            33        1,183    34,577        29.2
AeP            13          439    13,802        31.4
B               4          290     4,711        16.2
BC             16        1,890    16,325         8.6
DC             14          895    15,048        16.8
E               7          351     7,810        22.3
FB             24        1,882    24,963        13.3
GAp            13          953    14,034        14.7
GAr            26        1,264    26,532        21
GB             11        1,287    11,720         9.1
GC             27        1,434    28,367        19.8
GM              7          441     7,936        18
GP             33        1,197    33,753        28.2
GVS            11          947    11,608        12.3
GiP            13          746    13,405        18
H               4          300     4,625        15.4

In [13]:
# Check authors with fewer than 5000 total tokens to make sure the splitting is working as expected
for author in train_1000.get_unique_values("author"):
    author_corpus = train_1000.filter_by_level("author", author)
    author_tokens = sum(len(s.text.split()) for s in author_corpus)
    if author_tokens < 5000:
        print(f"\n{author}: {author_tokens} tokens")
        for work_name in author_corpus.get_unique_values("work"):
            work_corpus = author_corpus.filter_by_level("work", work_name)
            work_tokens = sum(len(s.text.split()) for s in work_corpus)
            print(f"  {work_name}: {work_tokens} tokens")


B: 4711 tokens
  judas_1: 1167 tokens
  judas_2: 1188 tokens
  judas_3: 1181 tokens
  judas_4: 1175 tokens

H: 4625 tokens
  maria_1: 1162 tokens
  maria_2: 1141 tokens
  maria_3: 1158 tokens
  maria_4: 1164 tokens

RT: 4440 tokens
  mauri_1: 1097 tokens
  mauri_2: 1090 tokens
  mauri_3: 1102 tokens
  mauri_4: 1151 tokens

SG: 1722 tokens
  ecclesiasten_1: 1722 tokens

U1: 1617 tokens
  canticamariae_1: 1617 tokens

U2: 2244 tokens
  liberesdre_1: 1125 tokens
  liberesdre_2: 1119 tokens


## Stage 4: Save Corpora

Pickle corpora for later use.

In [15]:
with open("train_corpus.pkl", "wb") as f:
    pickle.dump(train_corpus, f)

with open("test_corpus.pkl", "wb") as f:
    pickle.dump(test_corpus, f)

with open("machabeis_corpus.pkl", "wb") as f:
    pickle.dump(machabeis_corpus, f)

print("Corpora pickled successfully.")
print("Files created: train_corpus.pkl, test_corpus.pkl, machabeis_corpus.pkl")

Corpora pickled successfully.
Files created: train_corpus.pkl, test_corpus.pkl, machabeis_corpus.pkl
